In [ ]:
import polars as pl
import polars.selectors as cs
import time
import os

## Using Jupyter

If this is your first time in JupyterLab, here are the basics you need:

| Action | How |
|--------|-----|
| Run a cell and move to the next | **Shift + Enter** |
| Run a cell and stay | **Ctrl + Enter** (macOS: **Cmd + Enter**) |
| Edit a markdown cell | Double-click it |
| Render a markdown cell back | **Shift + Enter** |
| Insert cell above / below | Press **Esc** to enter command mode, then **A** / **B** |


## Graphviz setup

`show_graph()` (used later today) needs two things:

1. The `graphviz` Python package - already in this project's `pyproject.toml`.
2. The **Graphviz system binary**:
   - macOS: `brew install graphviz`
   - Linux: `sudo apt-get install graphviz`
   - Windows: [graphviz.org/download](https://graphviz.org/download/)

Run the cell below if you are on Linux and graphviz is not yet installed.


In [ ]:
! sudo apt-get install -y graphviz


# Python Polars Essential Training - Day 1

Welcome! Today you'll learn the foundation of using Polars. We'll go over all the base data structures and core concepts of Polars so you understand how to build your own queries.

Let's start with importing Polars and printing the versions to check if everything works as it should.

In [ ]:
pl.show_versions()

## Base Data Structures - Exercises

In the cell below, create 3 Series objects. Create one containing some strings you make up, some ints, and some floats.

See [pl.Series](https://docs.pola.rs/api/python/dev/reference/series/index.html).

In [ ]:
# Your code here

Now create a DataFrame out of these Series.

See [pl.DataFrame](https://docs.pola.rs/api/python/dev/reference/dataframe/index.html).

In [ ]:
# Your code here

Create a DataFrame straight from a Python dictionary.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Creating Series</summary>

> ```python
> series_names = pl.Series("name", ["Alice", "Bob", "Charlie"])
> series_ages = pl.Series("age", [25, 30, 35])
> series_scores = pl.Series("score", [85.5, 90.0, 78.5])
> ```
</details>

<details>
<summary>Show Answer: DataFrame from Series</summary>

> ```python
> df_from_series = pl.DataFrame([series_names, series_ages, series_scores])
> df_from_series
> ```
</details>

<details>
<summary>Show Answer: DataFrame from Dictionary</summary>

> ```python
> df_from_dict = pl.DataFrame(
>     {
>         "name": ["David", "Eve", "Frank"],
>         "age": [40, 45, 50],
>         "score": [92.0, 88.5, 76.0],
>     }
> )
> df_from_dict
> ```
</details>


#### End of section, wait here


## Input / Output

Writing all your data yourself takes a while and presumably you want to work on data that already exists.
This is where the suite of IO-related methods comes in.

In the cells below, read and display `data/scientists.xlsx`.
See [pl.read_excel](https://docs.pola.rs/api/python/dev/reference/api/polars.read_excel.html).


In [ ]:
# Your code here

Load and display `data/animals_dirty.csv`.

**Hint**: This file uses `;` as separator, has a junk row after the header, and uses `"n/a"`, `"-"`, and `"NA"` as null values.
See [pl.read_csv](https://docs.pola.rs/api/python/dev/reference/api/polars.read_csv.html).


In [ ]:
# Your code here

### Benchmarking

Throughout this course we'll compare the performance of different approaches. Here's a simple benchmark function
that runs code multiple times and returns the **minimum** execution time. We use `min()` because
system noise (GC, OS scheduling, fan ramps) only ever makes a run *slower*, never faster - the minimum is the
closest estimate of intrinsic speed.

**Usage**: Wrap your code in a lambda and pass it to `benchmark()`:
```python
result = benchmark(lambda: pl.read_parquet("data/file.parquet"))
print(f"Took {result:.3f}s")

time_a = benchmark(lambda: approach_a())
time_b = benchmark(lambda: approach_b())
print(f"A: {time_a:.3f}s, B: {time_b:.3f}s")
```


In [ ]:
def benchmark(func, runs=5):
    # runs=5, return min: system noise only ever makes a run slower, min best estimates intrinsic speed
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        func()
        times.append(time.perf_counter() - start)
    return min(times)


Now let's transform parquet data into CSV. Compare:

- **Eager**: `pl.read_parquet(...).select(...).filter(...).write_csv(...)`
- **Lazy**: `pl.scan_parquet(...).select(...).filter(...).sink_csv(...)`

Use the glob `data/yellow_tripdata_2025-*.parquet` (both months, ~110 MB combined). Select only
`tpep_pickup_datetime`, `passenger_count`, `tip_amount`, `total_amount`. Filter for `passenger_count >= 1`.

The eager approach reads all 20+ columns into memory before selecting and filtering. The lazy approach
pushes the selection and filter into the scan, reading only what it needs.

See [pl.scan_parquet](https://docs.pola.rs/api/python/dev/reference/api/polars.scan_parquet.html),
[LazyFrame.sink_csv](https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.sink_csv.html),
[DataFrame.write_csv](https://docs.pola.rs/api/python/dev/reference/api/polars.DataFrame.write_csv.html).


In [ ]:
# Your code here

### Time for the difference: Formats matter!

Let's see what the difference is in reading a CSV file vs a Parquet file.

First create `data/yellow_tripdata_2025-01.csv` by reading the parquet file and writing it to CSV.
Then use `benchmark()` to compare reading `yellow_tripdata_2025-01.parquet` vs `yellow_tripdata_2025-01.csv`.

Also compare the file sizes with `os.path.getsize("filepath")`.

See [pl.read_parquet](https://docs.pola.rs/api/python/dev/reference/api/polars.read_parquet.html),
[pl.read_csv](https://docs.pola.rs/api/python/dev/reference/api/polars.read_csv.html).


In [ ]:
# Your code here

In order to inspect the effect of the filetype on the data you've read, you can call `.schema` or `.dtypes` on the DataFrame. What's the difference between the schemas from a DataFrame that comes from parquet compared to a DataFrame that comes from csv?

In [ ]:
# Your code here

As you can see, the CSV reader infers all columns as strings or generic types. You can use `.cast()` to convert columns to the correct type. Try casting the `tpep_pickup_datetime` and `tpep_dropoff_datetime` columns from the CSV DataFrame to `pl.Datetime`, and the `passenger_count` column to `pl.Int64`.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Reading Excel Files</summary>

> ```python
> scientists = pl.read_excel("data/scientists.xlsx")
> scientists
> ```
</details>

<details>
<summary>Show Answer: Reading CSV with Options</summary>

> ```python
> animals_dirty = pl.read_csv(
>     "data/animals_dirty.csv",
>     separator=";",
>     skip_rows_after_header=1,
>     null_values=["n/a", "-", "NA"],
> )
> animals_dirty
> ```
</details>

<details>
<summary>Show Answer: Parquet to CSV Conversion</summary>

> ```python
> COLS = ["tpep_pickup_datetime", "passenger_count", "tip_amount", "total_amount"]
>
> rw = benchmark(lambda: (
>     pl.read_parquet("data/yellow_tripdata_2025-*.parquet")
>     .select(COLS)
>     .filter(pl.col("passenger_count") >= 1)
>     .write_csv("data/yellow_tripdata_combined.csv")
> ))
> print(f"Read/Write time: {rw:.3f}s")
>
> ss = benchmark(lambda: (
>     pl.scan_parquet("data/yellow_tripdata_2025-*.parquet")
>     .select(COLS)
>     .filter(pl.col("passenger_count") >= 1)
>     .sink_csv("data/yellow_tripdata_combined.csv")
> ))
> print(f"Scan/Sink time:  {ss:.3f}s")
> print(f"Lazy is {rw / ss:.1f}x faster!")
> ```
</details>

<details>
<summary>Show Answer: Format Comparison Benchmark</summary>

> ```python
> # First, create the CSV file
> pl.read_parquet("data/yellow_tripdata_2025-01.parquet").write_csv("data/yellow_tripdata_2025-01.csv")
>
> # Benchmark reading both formats
> parquet_time = benchmark(lambda: pl.read_parquet("data/yellow_tripdata_2025-01.parquet"))
> csv_time = benchmark(lambda: pl.read_csv("data/yellow_tripdata_2025-01.csv"))
>
> print(f"Parquet: {parquet_time:.3f}s, CSV: {csv_time:.3f}s")
> print(f"Parquet is {csv_time / parquet_time:.1f}x faster")
> print(f"Parquet size: {os.path.getsize('data/yellow_tripdata_2025-01.parquet') / 1024 / 1024:.1f} MB")
> print(f"CSV size:     {os.path.getsize('data/yellow_tripdata_2025-01.csv') / 1024 / 1024:.1f} MB")
> ```
</details>

<details>
<summary>Show Answer: Schema Inspection</summary>

> ```python
> taxi_parquet = pl.read_parquet("data/yellow_tripdata_2025-01.parquet")
> taxi_csv = pl.read_csv("data/yellow_tripdata_2025-01.csv")
> taxi_parquet.schema, taxi_csv.schema
> ```
</details>

<details>
<summary>Show Answer: Casting Types</summary>

> ```python
> taxi_csv_casted = taxi_csv.with_columns(
>     pl.col("tpep_pickup_datetime").cast(pl.Datetime),
>     pl.col("tpep_dropoff_datetime").cast(pl.Datetime),
>     pl.col("passenger_count").cast(pl.Int64),
> )
> taxi_csv_casted.schema
> ```
</details>


Run the cells below to load a clean version of the animals and taxi dataset for the rest of the notebook.


In [ ]:
animals = pl.read_csv("data/animals.csv")


In [ ]:
taxi_parquet = pl.read_parquet("data/yellow_tripdata_2025-01.parquet")
taxi_csv = pl.read_csv("data/yellow_tripdata_2025-01.csv")

### Bonus: Cloud Storage

Real-world data often lives in the cloud (S3, Azure Blob, GCS).
Polars supports reading [directly from cloud storage](https://docs.pola.rs/user-guide/io/cloud-storage/).

```python
# Example (requires authentication)
pl.read_parquet(
    "s3://my-bucket/data.parquet",
    storage_options={
        "aws_access_key_id": "...",
        "aws_secret_access_key": "...",
        "aws_region": "us-east-1",
    }
)
```

This works for both `read_*` (eager) and `scan_*` (lazy) operations.


## Lazy vs Eager execution

Polars has two execution modes: lazy and eager.

- **Eager** (`read_parquet`, `write_parquet`, etc.): code executes immediately, all data lands in memory.
- **Lazy** (`scan_parquet`, `sink_parquet`, etc.): Polars builds a query plan and only executes when you call `.collect()` or `sink_*()`.

The lazy engine can **push predicates and projections into the scan**, reading only the rows and columns
it actually needs. This makes a large difference on wide tables.

Use `benchmark()` to compare lazy and eager execution. Use the glob `data/yellow_tripdata_2025-*.parquet`
(both months). Select columns `VendorID`, `passenger_count`, `tip_amount` and filter for `pl.col(passenger_count) > 2`.

Compare `pl.read_parquet` + chain vs `pl.scan_parquet` + chain + `.collect()`.

See [pl.scan_parquet](https://docs.pola.rs/api/python/dev/reference/api/polars.scan_parquet.html),
[LazyFrame.collect](https://docs.pola.rs/api/python/dev/reference/lazyframe/api/polars.LazyFrame.collect.html),
[Lazy API user guide](https://docs.pola.rs/user-guide/lazy/query-plan/).


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Lazy vs Eager Execution</summary>

> ```python
> eager_time = benchmark(lambda: (
>     pl.read_parquet("data/yellow_tripdata_2025-*.parquet")
>     .select("VendorID", "passenger_count", "tip_amount")
>     .filter(pl.col("passenger_count") > 2)
> ))
>
> lazy_time = benchmark(lambda: (
>     pl.scan_parquet("data/yellow_tripdata_2025-*.parquet")
>     .select("VendorID", "passenger_count", "tip_amount")
>     .filter(pl.col("passenger_count") > 2)
>     .collect()
> ))
>
> print(f"Eager: {eager_time:.3f}s, Lazy: {lazy_time:.3f}s")
> print(f"Lazy is {eager_time / lazy_time:.1f}x faster!")
> ```
</details>


#### End of section, wait here


## Contexts

Now that we've got some data to work with, it's time to start manipulating it.
The main contexts in Polars are `select`, `with_columns`, `filter`, `group_by` with `agg`, and `join`.
Let's explore them one by one.

### Select

`select` is used to select columns from a DataFrame.
Select the `Name` and `Born` columns from the `scientists` DataFrame.

See [DataFrame.select](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.select.html).


In [ ]:
# Your code here

### With Columns

`with_columns` is used to add or modify columns.
Add a column `age` to the `scientists` DataFrame. You can calculate it by taking the `Died` column and subtracting the `Born` column.

See [DataFrame.with_columns](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.with_columns.html).


In [ ]:
# Your code here

### Filter

`filter` is used to select rows based on a condition.
Filter the `scientists` DataFrame to only include scientists born after 1900.

See [DataFrame.filter](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.filter.html).


In [ ]:
# Your code here

### Group By and Agg

`group_by` is used to group rows that have the same values in specified columns into a single row.
`agg` is used to aggregate the results of the group by.

Let's use the `animals` DataFrame (loaded from `data/animals.csv` via the safety-net cell above).
Group by `Conservation_Status` and count the number of animals in each status.

See [DataFrame.group_by](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.group_by.html),
[GroupBy.agg](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html).


In [ ]:
# Your code here

### Join

Real-world data often comes from multiple sources. We have a file `data/awards.csv` that contains the number of awards won by various scientists.

Your task:
1. Load `data/awards.csv`.
2. Perform a **left join** to combine it with your `scientists` DataFrame. We want to keep all scientists, even if they don't have an entry in the awards file.
3. Clean up: The join will result in `null` values for scientists not found in the awards file. Fill these `null`s with `0`.

See [DataFrame.join](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.join.html).


In [ ]:
# Your code here

### Selectors

Typing out every column name is for people with too much time. Use `polars.selectors` (imported as `cs`) to select columns by type, name patterns, or other properties.

Tasks using the `animals` DataFrame:
1. Select all numeric columns.
2. Select the `Animal` column.
3. Select all columns that contain the word "Lifespan".

See [polars.selectors](https://docs.pola.rs/api/python/dev/reference/selectors.html).


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Select</summary>

> ```python
> scientists.select(["Name", "Born"])
> ```
</details>

<details>
<summary>Show Answer: With Columns</summary>

> ```python
> scientists.with_columns((pl.col("Died") - pl.col("Born")).alias("age"))
> ```
</details>

<details>
<summary>Show Answer: Filter</summary>

> ```python
> scientists.filter(pl.col("Born") > 1900)
> ```
</details>

<details>
<summary>Show Answer: Group By and Agg</summary>

> ```python
> animals.group_by("Conservation_Status").agg(pl.len().alias("count"))
> ```
</details>

<details>
<summary>Show Answer: Join</summary>

> ```python
> awards = pl.read_csv("data/awards.csv")
> scientists.join(awards, on="Name", how="left").with_columns(
>     pl.col("Major_Awards").fill_null(0)
> )
> ```
</details>

<details>
<summary>Show Answer: Selectors</summary>

> ```python
> # 1. Select all numeric columns
> print(animals.select(cs.numeric()))
> 
> # 2. Select the Animal column
> print(animals.select("Animal"))
> 
> # 3. Select all columns that contain "Lifespan"
> print(animals.select(cs.contains("Lifespan")))
> ```
</details>

#### End of section, wait here


## Reshaping DataFrames

DataFrames often need to be reshaped to fit the needs of your analysis. Two common formats are:

Polars provides several methods to transform the shape of your data: `pivot`, `unpivot`, `transpose`, and `explode`.

### Pivot (Long to Wide)

`pivot` transforms data from long to wide format. It takes values from one column and spreads them across multiple columns.

Using `taxi_parquet` (the January taxi data from `data/yellow_tripdata_2025-01.parquet`, loaded as `taxi_parquet` in the I/O answers above), create a summary showing the **average trip distance** for each `VendorID` broken down by `payment_type`. The result should be a wide DataFrame where:
- Each row is a `VendorID`
- Each column (besides VendorID) is a payment type
- Values are the average `trip_distance` for that vendor/payment combination

**Hint**: You'll need to aggregate first, then pivot. See [DataFrame.pivot](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.pivot.html).


In [ ]:
# Your code here

### Unpivot (Wide to Long)

`unpivot` is the opposite of pivot: it transforms data from wide to long format. This is also known as "melting" a DataFrame.

The `animals` DataFrame has two numeric measurement columns: `Average_Weight_kg` and `Average_Lifespan_years`. Unpivot these into a long format where you have:
- `Animal` and `Conservation_Status` as identifier columns
- A `measurement` column containing either "Average_Weight_kg" or "Average_Lifespan_years"
- A `value` column containing the numeric value

Then group by `Conservation_Status` and `measurement`, and compute the mean value per group.

See [DataFrame.unpivot](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.unpivot.html).


In [ ]:
# Your code here

### Transpose

`transpose` flips rows and columns: each row becomes a column and each column becomes a row.

Using the `animals` DataFrame:
1. First, select only the numeric columns.
2. Calculate summary statistics using `.describe()`.
3. Transpose the result so that each statistic becomes a column and each original numeric column becomes a row.

See [DataFrame.transpose](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.transpose.html).


In [ ]:
# Your code here

### Explode

`explode` expands list columns into separate rows. Each element in the list becomes its own row.

Here's a simple example: we have orders where each order contains multiple items:

See [DataFrame.explode](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.explode.html).


In [ ]:
orders = pl.DataFrame({
    "order_id": [1, 2, 3],
    "customer": ["Alice", "Bob", "Alice"],
    "items": [["apple", "banana", "orange"], ["apple", "grape"], ["banana", "grape", "melon"]],
})
orders

1. Explode the `items` column so each item gets its own row
2. Count how many times each item was ordered across all orders
3. Find the most popular item

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Pivot</summary>

> ```python
> (
>     taxi_parquet
>     .group_by("VendorID", "payment_type")
>     .agg(pl.col("trip_distance").mean().alias("avg_distance"))
>     .pivot(on="payment_type", index="VendorID", values="avg_distance")
> )
> ```
</details>

<details>
<summary>Show Answer: Unpivot</summary>

> ```python
> (
>     animals
>     .unpivot(
>         index=["Animal", "Conservation_Status"],
>         on=["Average_Weight_kg", "Average_Lifespan_years"],
>         variable_name="measurement",
>         value_name="value",
>     )
>     .group_by("Conservation_Status", "measurement")
>     .agg(pl.col("value").mean().alias("mean_value"))
>     .sort("Conservation_Status", "measurement")
> )
> ```
</details>

<details>
<summary>Show Answer: Transpose</summary>

> ```python
> (
>     animals
>     .select(cs.numeric())
>     .describe()
>     .transpose(include_header=True, header_name="metric", column_names="statistic")
> )
> ```
</details>

<details>
<summary>Show Answer: Explode</summary>

> ```python
> (
>     orders
>     .explode("items")
>     .group_by("items")
>     .len()
>     .sort("len", descending=True)
> )
> ```
</details>

#### End of section, wait here


## Expressions

Expressions are the heart of Polars. They are used to perform operations on columns.
You've already used them in the previous exercises!
`pl.col("Born") > 1900` is an expression.
`pl.col("Died") - pl.col("Born")` is an expression.

Let's try a more complex one.
From the `animals` dataset, create a new column `Lifespan_Diff_From_Mean` which is the `Average_Lifespan_years` minus the average lifespan of all animals in the dataset.

See the [Expressions user guide](https://docs.pola.rs/user-guide/expressions/expression-expansion/).


In [ ]:
# Your code here

### Namespaces

Polars expressions have specialized namespaces for specific data types.
- `.str` for string operations
- `.dt` for datetime operations
- `.list` for list operations
- `.struct` for struct operations

You can access these namespaces on a column expression. For example: `pl.col("name").str.to_uppercase()`.

Using the `animals` DataFrame:
1. Create a new column `Animal_Upper` which is the `Animal` name in uppercase.
2. Create a new column `Genus` which is the first word of the `Scientific_Name` column.

See [Expr.str](https://docs.pola.rs/api/python/dev/reference/expressions/string.html),
[Expr.list](https://docs.pola.rs/api/python/dev/reference/expressions/list.html).


In [ ]:
# Your code here

### Datetime Operations

Polars has first-class support for datetimes via the `.dt` namespace.

Using `taxi_parquet` (the January taxi data from `data/yellow_tripdata_2025-01.parquet`, loaded as `taxi_parquet` in the I/O answers above):
1. Create a new column `pickup_hour` that extracts the hour from `tpep_pickup_datetime`.
2. Create a new column `day_of_week` that extracts the day of the week (Monday=1, Sunday=7).

See [Expr.dt](https://docs.pola.rs/api/python/dev/reference/expressions/temporal.html).


In [ ]:
# Your code here

### Logic: When / Then / Otherwise

To perform conditional logic, use `when().then().otherwise()`.
This is similar to if-else statements or SQL CASE WHEN.

Syntax:
```python
pl.when(condition) \
.then(value_if_true) \
.otherwise(value_if_false)
```

You can chain multiple `.when().then()` calls.

Using the `animals` DataFrame, create a `Lifespan_Category` column:
- "Short" if `Average_Lifespan_years` < 15
- "Medium" if `Average_Lifespan_years` is between 15 and 25 (inclusive)
- "Long" if `Average_Lifespan_years` > 25
- "Unknown" if the value is null

See [pl.when](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.when.html).


In [ ]:
# Your code here

### Nulls and Sorting

Real world data is often messy and contains missing values (Nulls).
Polars provides methods to handle them:
- `fill_null(value)`: Replace nulls with a value.
- `drop_nulls()`: Remove rows with nulls.

Sorting is done with `.sort("column_name", descending=True/False)`.

Using the `animals` DataFrame:
1. Fill the null values in `Average_Weight_kg` with the mean weight of all animals.
2. Sort the result by `Average_Weight_kg` in descending order.

See [Expr.fill_null](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.fill_null.html),
[DataFrame.sort](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.sort.html).


In [ ]:
# Your code here

## Window Functions

Window functions allow you to perform aggregations without grouping the entire DataFrame. This is done using the `.over()` expression.
It's like a "group_by, agg, and join back" all in one line.

Tasks using the `animals` DataFrame:
1. Create a column `Mean_Lifespan_By_Status` which is the average lifespan *per* `Conservation_Status`.
2. Create a column `Rank_Weight_By_Status` which ranks the animals by `Average_Weight_kg` within each `Conservation_Status`.

See [Expr.over](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.over.html).


In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Expressions</summary>

> ```python
> animals.with_columns(
>     (pl.col("Average_Lifespan_years") - pl.col("Average_Lifespan_years").mean()).alias("Lifespan_Diff_From_Mean")
> )
> ```
</details>

<details>
<summary>Show Answer: Namespaces</summary>

> ```python
> animals.with_columns(
>     pl.col("Animal").str.to_uppercase().alias("Animal_Upper"),
>     pl.col("Scientific_Name").str.split(" ").list.get(0).alias("Genus"),
> )
> ```
</details>

<details>
<summary>Show Answer: Datetime Operations</summary>

> ```python
> taxi_parquet.with_columns(
>     pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
>     pl.col("tpep_pickup_datetime").dt.weekday().alias("day_of_week"),
> ).select("tpep_pickup_datetime", "pickup_hour", "day_of_week").sample(5)
> ```
</details>

<details>
<summary>Show Answer: Logic: When / Then / Otherwise</summary>

> ```python
> animals.with_columns(
>     pl.when(pl.col("Average_Lifespan_years") < 15)
>     .then(pl.lit("Short"))
>     .when(pl.col("Average_Lifespan_years") <= 25)
>     .then(pl.lit("Medium"))
>     .when(pl.col("Average_Lifespan_years") > 25)
>     .then(pl.lit("Long"))
>     .otherwise(pl.lit("Unknown"))
>     .alias("Lifespan_Category")
> )
> ```
</details>

<details>
<summary>Show Answer: Nulls and Sorting</summary>

> ```python
> animals.with_columns(
>     pl.col("Average_Weight_kg").fill_null(pl.mean("Average_Weight_kg"))
> ).sort("Average_Weight_kg", descending=True)
> ```
</details>

<details>
<summary>Show Answer: Window Functions</summary>

> ```python
> animals.with_columns(
>     pl.col("Average_Lifespan_years").mean().over("Conservation_Status").alias("Mean_Lifespan_By_Status"),
>     pl.col("Average_Weight_kg").rank(descending=True).over("Conservation_Status").alias("Rank_Weight_By_Status"),
> ).select("Animal", "Conservation_Status", "Mean_Lifespan_By_Status", "Rank_Weight_By_Status")
> ```
</details>


#### End of section, wait here


## Case study: Yellow taxi dataset

Let's use the yellow taxi dataset to do some analysis across two months of data.

Files:
- `data/yellow_tripdata_2025-01.parquet` (January 2025)
- `data/yellow_tripdata_2025-02.parquet` (February 2025)

Calculate the average tip percentage `(tip_amount / total_amount) * 100` for every `VendorID`,
and find out which Vendor had the biggest improvement in tip percentage going from January to February.

See [pl.scan_parquet](https://docs.pola.rs/api/python/dev/reference/api/polars.scan_parquet.html), [DataFrame.group_by](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.group_by.html), and [DataFrame.join](https://docs.pola.rs/api/python/dev/reference/dataframe/api/polars.DataFrame.join.html).

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Morning Capstone</summary>

> ```python
> # Create a function we can apply on both frames
> def vendor_tip_percentage(path: str, month: str) -> pl.LazyFrame:
>     return (
>         pl.scan_parquet(path)
>         .select(
>             pl.col("VendorID"),
>             pl.col("total_amount"),
>             pl.col("tip_amount"),
>         )
>         .group_by("VendorID")
>         .agg(
>             (
>                 pl.col("tip_amount").sum() / pl.col("total_amount").sum().replace(0, None) * 100
>             ).alias(f"tip_percentage_{month}")
>         )
>     )
> 
> 
> # Process each month independently
> jan = vendor_tip_percentage(
>     "data/yellow_tripdata_2025-01.parquet",
>     "january",
> )
> 
> feb = vendor_tip_percentage(
>     "data/yellow_tripdata_2025-02.parquet",
>     "february",
> )
> 
> # Join on VendorID, compute the increase and filter for the max
> result = (
>     jan.join(feb, on="VendorID", how="inner")
>     .with_columns(
>         (
>             pl.col("tip_percentage_february")
>             - pl.col("tip_percentage_january")
>         ).alias("increase")
>     )
>     .filter(pl.col("increase") == pl.col("increase").max())
> )
> 
> result.collect()
> ```
</details>
